# **TASK 10**

For Task 10, I initially had issues because the main notebook was written in normal Python, while the Lakehouse table code required Spark. This caused errors such as spark is not defined.

To avoid changing the whole notebook to PySpark and affecting the rest of the team’s code, I kept All the Tasks in the original Python notebook. I then saved the df_results DataFrame from Task 8 as a CSV file into the Lakehouse Files area using:

df_results.to_csv("/lakehouse/default/Files/df_results_task10.csv", index=False)

After that, I used a separate notebook connected to the same Lakehouse to read the CSV file with Spark 

This fixed the issue because the modelling work stayed in Python, while only the final Lakehouse table creation was completed using Spark.

In [1]:
import pandas as pd

# Initialize df_results with the saved table (Actual vs Predicted values)
df_results = pd.read_csv("/lakehouse/default/Files/df_results_task10.csv")

# Rename column names by replacing characters - " ", "(", ")" with underscore or empty string to avoid AnalysisException Error in the following PySpark code operation  
df_results.columns = df_results.columns.str.replace(' ', '_')
df_results.columns = df_results.columns.str.replace('(', '')
df_results.columns = df_results.columns.str.replace(')', '')

display(df_results.head(20))

StatementMeta(, 3ef2a947-4cf3-47d0-a625-79066a022b9d, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a93fb777-239d-4f1c-9b35-a7e778cb3733)

In [2]:
df_results_spark = spark.createDataFrame(df_results)

df_results_spark.createOrReplaceTempView("df_results_spark_view")

df_spark= spark.table("df_results_spark_view")

# Write to lakehouse as a table in the number write the number of your team.
df_spark.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("Team2_Mission5_Table")

StatementMeta(, 3ef2a947-4cf3-47d0-a625-79066a022b9d, 5, Finished, Available, Finished, False)

##### Check Team2_Mission5_Table by displaying the first 20 rows to see if the table with Actual vs Predicted values is correctly saved in the lakehouse

In [3]:
# Read the saved table into a Spark Dataframe
df_spark_table = spark.read.table("Team2_Mission5_Table")

# Display the first 20 rows of the saved table
df_spark_table.show(20, truncate=False)

StatementMeta(, 3ef2a947-4cf3-47d0-a625-79066a022b9d, 6, Finished, Available, Finished, False)

+-----------+----------------------------------+----------------------------+--------------------------------+
|Actual_Loan|Predicted_Loan_Logistic_Regression|Predicted_Loan_Random_Forest|Predicted_Loan_Gradient_Boosting|
+-----------+----------------------------------+----------------------------+--------------------------------+
|0          |0                                 |0                           |0                               |
|1          |1                                 |1                           |1                               |
|0          |0                                 |0                           |0                               |
|0          |0                                 |0                           |0                               |
|0          |0                                 |0                           |0                               |
|1          |1                                 |1                           |1                               |
|